# 20 - USS-prompted traversability, step 0: map objects in the image

Method-paper feasibility test (CVPR 2027 track). For every labelled frame of
the paper's dataset (r30-85_n20 window), record every production USS map
object (`Map_Obj{n}_P1/P2`, vehicle frame) alive at that frame's MF4 time.
A local script (`scripts/uss_prompt_analysis.py`) then fits one ground-plane
pinhole camera per bumper and measures how often the USS object projects
onto the obstacle's image box - including on sequences the detector misses.

CPU only. Resumable (status CSV = done set). Smoke with `MAX_SEQUENCES`.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import csv
import json as _json
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import asammdf  # fail HERE, loudly, if the MF4 reader cannot import
print(f"asammdf {asammdf.__version__}")
from asammdf import MDF

from vision_uss_research.alignment.mf4 import (find_camera_channel, sorted_pairs,
                                               video_to_mf4)
from vision_uss_research.method.uss_prompt import (read_slots, slots_at,
                                                   window_video_seconds)
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import sequence_files, with_drive_retry

DATASET_RUN = "20260718_dataset_v2"   # the paper's default split (test = 963)
MAX_SEQUENCES = 50          # smoke first; 0 = all test + N_TRAIN train sequences
N_TRAIN = 800               # train-split sequences for the camera fit
NUM_WORKERS = 8
SEED = 0
MAX_DT_S = 0.15             # slot sample must be within this of the frame time

RUN_DIR = ARTIFACTS / "method" / new_run_id("prompt", "uss_proj_v1")
SLOTS_CSV = RUN_DIR / "slots_at_frames.csv"
STATUS_CSV = RUN_DIR / "sequence_status.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SLOT_FIELDS = ["sequence_id", "camera", "frame_idx", "t_mf4", "slot", "dt",
               "p1x", "p1y", "p2x", "p2y", "mid_x", "mid_y"]
STATUS_FIELDS = ["sequence_id", "camera", "split", "error", "camera_channel",
                 "duration_s", "n_frames", "n_frames_with_slots", "n_slots_in_mf4"]

man = pd.read_csv(ARTIFACTS / "datasets" / DATASET_RUN / "dataset_manifest.csv",
                  usecols=["split", "sequence_id", "camera"])
pairs = man.drop_duplicates(["sequence_id", "camera"])
test = pairs[pairs.split == "test"]
train_seqs = (pairs[pairs.split == "train"].drop_duplicates("sequence_id")
              .sample(frac=1, random_state=SEED).sequence_id.head(N_TRAIN))
todo_pairs = pd.concat([test, pairs[pairs.sequence_id.isin(train_seqs)]])
if MAX_SEQUENCES:
    keep = todo_pairs.drop_duplicates("sequence_id").sequence_id.sample(
        frac=1, random_state=SEED).head(MAX_SEQUENCES)
    todo_pairs = todo_pairs[todo_pairs.sequence_id.isin(keep)]

done = set()
if STATUS_CSV.exists():
    st = pd.read_csv(STATUS_CSV, dtype=str).fillna("")
    done = set(st.loc[st.error == "", "sequence_id"])
todo = [s for s in todo_pairs.sequence_id.unique() if s not in done]
start_run(RUN_DIR, config={"dataset_run": DATASET_RUN, "max_sequences": MAX_SEQUENCES,
                           "n_train": N_TRAIN, "max_dt_s": MAX_DT_S},
          inputs=[f"datasets/{DATASET_RUN}"], repo_root=REPO_DIR)
print(f"sequences planned {todo_pairs.sequence_id.nunique()} | done {len(done)} | todo {len(todo)}")
print(f"run dir: {RUN_DIR}")

## Collect map objects at every labelled frame

In [ ]:
def video_duration(path):
    cap = cv2.VideoCapture(str(path))
    n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return n / fps if fps else None

def process(seq_id):
    """All cameras labelled for this sequence; returns (status_rows, slot_rows)."""
    seq_dir = RAW_ROOT / seq_id
    cams = todo_pairs.loc[todo_pairs.sequence_id == seq_id]
    status, rows = [], []
    try:
        found = with_drive_retry(sequence_files, seq_dir)
        mf4 = next(iter(sorted(seq_dir.glob("*.mf4"))), None)
        if mf4 is None:
            raise FileNotFoundError("no_mf4")
        mdf = MDF(str(mf4))
        try:
            names = set(mdf.channels_db)
            slots = read_slots(mdf)
            for _, c in cams.iterrows():
                cam, split = c.camera, c.split
                st = {"sequence_id": seq_id, "camera": cam, "split": split, "error": "",
                      "n_slots_in_mf4": len(slots)}
                video = found.get(f"{cam}_video")
                ch = find_camera_channel(names, cam)
                dur = video_duration(video) if video else None
                if video is None or ch is None or not dur:
                    st["error"] = "no_video_or_camera_channel"
                    status.append(st); continue
                sig = mdf.get(ch)
                pts, mt = sorted_pairs(sig.samples, sig.timestamps)
                t_frames = video_to_mf4(window_video_seconds(dur), pts, mt)
                with_slot = 0
                for i, t in enumerate(t_frames):
                    live = slots_at(slots, float(t), MAX_DT_S)
                    with_slot += bool(live)
                    rows += [{"sequence_id": seq_id, "camera": cam, "frame_idx": i,
                              "t_mf4": round(float(t), 3), **{k: (round(v, 3) if isinstance(v, float) else v)
                                                               for k, v in r.items()}}
                             for r in live]
                st.update(camera_channel=ch, duration_s=round(dur, 2),
                          n_frames=len(t_frames), n_frames_with_slots=with_slot)
                status.append(st)
        finally:
            mdf.close()
    except Exception as e:
        status = [{"sequence_id": seq_id, "camera": c, "split": s,
                   "error": f"{type(e).__name__}:{e}"[:200]}
                  for c, s in cams[["camera", "split"]].itertuples(index=False)]
        rows = []
    return status, rows

def append(path, rows, fields):
    header = not path.exists()
    with path.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        if header:
            w.writeheader()
        w.writerows(rows)

# errored sequences are retried on resume: drop their old status rows first
if STATUS_CSV.exists():
    st = pd.read_csv(STATUS_CSV, dtype=str).fillna("")
    st[st.error == ""].to_csv(STATUS_CSV, index=False)

buf_s, buf_r = [], []
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futs = [pool.submit(process, s) for s in todo]
    for k, fut in enumerate(tqdm(as_completed(futs), total=len(futs), unit="seq"), 1):
        s, r = fut.result()
        buf_s += s; buf_r += r
        if k % 25 == 0 or k == len(futs):
            append(SLOTS_CSV, buf_r, SLOT_FIELDS); append(STATUS_CSV, buf_s, STATUS_FIELDS)
            buf_s, buf_r = [], []

## Summary

In [ ]:
st = pd.read_csv(STATUS_CSV).fillna({"error": ""})
ok = st[st.error == ""]
print(f"sequence-cameras: {len(st)} | ok {len(ok)} | errors {(st.error != '').sum()}")
print(st[st.error != ""].error.str.split(":").str[0].value_counts().head(8).to_string())
cov = ok.n_frames_with_slots / ok.n_frames
print(f"frames with >=1 live map object: {cov.mean():.3f} (median per seq {cov.median():.2f})")
print(ok.groupby("split").apply(lambda d: (d.n_frames_with_slots / d.n_frames).mean(),
                                include_groups=False).round(3).to_string())
finish_run(RUN_DIR, summary={"sequence_cameras": int(len(st)), "ok": int(len(ok)),
                             "frame_slot_coverage": round(float(cov.mean()), 4)})
print(f"\nnext: download {RUN_DIR.name}/ (slots_at_frames.csv + sequence_status.csv) "
      "and run scripts/uss_prompt_analysis.py locally")